# Langchain's Indexing API

What this means:
You're doing data ingestion into your vector databases.

### Note:
Recommended to run on **local computer** rather than on Colab because it involves also running a Docker command.


### Key takeaways:
	- Always add metadata to your langchain documents
	- You can have a RecordManager (a database) in any kind of SQL database that you want (Postgres, MySQL) - just change the db url and add connection strings with the namespace)
Pick the mode you are most interested in

In [ ]:
%pip install langchain langchain-openai langchain-community langchain-text-splitters langchain-chroma faiss-cpu --upgrade --quiet

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import CharacterTextSplitter
from langchain_community.vectorstores.faiss import FAISS

In [ ]:
import getpass
import os
import warnings

# Suppress LangChain deprecation warnings (we're intentionally using legacy chains)
warnings.filterwarnings("ignore", category=DeprecationWarning, module="langchain")

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

In [ ]:
raw_text = """
Digital marketing encompasses a broad range of marketing activities that utilize digital channels to connect with customers. At its core, digital marketing aims to reach a targeted audience through various online and electronic means, including social media, email, search engines, and websites. Unlike traditional marketing methods, digital marketing offers unparalleled opportunities for businesses to engage with their audience in real-time, enabling personalized communication and immediate feedback. This real-time interaction not only enhances customer experience but also allows businesses to gather valuable data on consumer behaviors, preferences, and trends, facilitating more effective marketing strategies and campaigns.
The rise of digital marketing can be attributed to the increasing reliance on the internet and digital devices by consumers. As more people spend time online, businesses have shifted their marketing efforts to where their audiences are. Digital marketing leverages this online presence, employing strategies such as search engine optimization (SEO), content marketing, pay-per-click (PPC) advertising, and social media marketing to improve visibility and attract potential customers. These strategies are designed to increase traffic to a company's online platforms, build brand awareness, and ultimately drive conversions and sales. The ability to measure the effectiveness of these strategies through analytics and metrics further underscores the advantage of digital marketing, enabling businesses to refine their approach and maximize return on investment (ROI).
"""

with open("test.txt", "w") as f:
    f.write(raw_text)

In [ ]:
# Load the document, split it into chunks, embed each chunk and load it into the vector store.
raw_documents = TextLoader('test.txt').load()

text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)

documents = text_splitter.split_documents(raw_documents)
db = FAISS.from_documents(documents, OpenAIEmbeddings())

In [ ]:
db.similarity_search_with_relevance_scores("digital marketing", k=1)

[(Document(id='0ec6c82e-ea67-4563-bbc4-4ca2e202f17f', metadata={'source': 'test.txt'}, page_content="Digital marketing encompasses a broad range of marketing activities that utilize digital channels to connect with customers. At its core, digital marketing aims to reach a targeted audience through various online and electronic means, including social media, email, search engines, and websites. Unlike traditional marketing methods, digital marketing offers unparalleled opportunities for businesses to engage with their audience in real-time, enabling personalized communication and immediate feedback. This real-time interaction not only enhances customer experience but also allows businesses to gather valuable data on consumer behaviors, preferences, and trends, facilitating more effective marketing strategies and campaigns.\nThe rise of digital marketing can be attributed to the increasing reliance on the internet and digital devices by consumers. As more people spend time online, busine

In [ ]:
# Adding on extra documents directly within LangChain:
from langchain_core.documents import Document

docs = [Document(page_content='James phoenix worked in digital marketing for 3 years.', metadata={'source': 'James Phoenix'}),
        Document(page_content='Digital marketing is a growing industry.', metadata={'source': 'Wikipedia'}),
        ]

In [ ]:
docs[0].metadata

{'source': 'James Phoenix'}

In [ ]:
docs[1].metadata

{'source': 'Wikipedia'}

In [ ]:
db.add_documents(docs)

['b89b0443-dac9-4665-8110-eee8ae01a3c4',
 'ce690f8b-8f5e-4ffb-8864-fcab3006e04f']

In [ ]:
db.similarity_search("James", k=1)

[Document(id='b89b0443-dac9-4665-8110-eee8ae01a3c4', metadata={'source': 'James Phoenix'}, page_content='James phoenix worked in digital marketing for 3 years.')]

---

## Indexing Becomes Messy Without A Record Manager

Without knowing which documents have been indexed, it is difficult to keep track of the documents that have been indexed. This is especially true when the documents are large and the indexing process is time-consuming.

LangChain solves this with an [Indexing API](https://python.langchain.com/docs/modules/data_connection/indexing) that exposes several different methods to keep track of the documents that have been indexed.

`docker run -p 9200:9200 -e "discovery.type=single-node" -e "xpack.security.enabled=false" -e "xpack.security.http.ssl.enabled=false" docker.elastic.co/elasticsearch/elasticsearch:8.9.0`

In [ ]:
from langchain_core.indexing import index, InMemoryRecordManager
from langchain_chroma import Chroma

collection_name = "test_index"
embedding = OpenAIEmbeddings()
vectorstore = Chroma(collection_name=collection_name, embedding_function=embedding)

In [ ]:
namespace = f"chroma/{collection_name}"
record_manager = InMemoryRecordManager(namespace=namespace)
record_manager.create_schema()

In [ ]:
updated_docs = [
    Document(
        page_content="James phoenix worked in digital marketing for 7 years.",
        metadata={"source": "James Phoenix"},
    ),

]

In [ ]:
def _clear():
    """Reset the local vector store and record manager so the indexing example can be rerun safely."""

    global vectorstore, record_manager
    vectorstore.delete_collection()
    vectorstore = Chroma(collection_name=collection_name, embedding_function=embedding)
    record_manager = InMemoryRecordManager(namespace=namespace)
    record_manager.create_schema()

In [ ]:
_clear()

In [ ]:
docs

[Document(metadata={'source': 'James Phoenix'}, page_content='James phoenix worked in digital marketing for 3 years.'),
 Document(metadata={'source': 'Wikipedia'}, page_content='Digital marketing is a growing industry.')]

In [ ]:
# Indexing all of the documents:
index(
    docs,
    record_manager,
    vectorstore,
    cleanup="incremental",
    source_id_key="source",
    key_encoder="blake2b",
)

{'num_added': 2, 'num_updated': 0, 'num_skipped': 0, 'num_deleted': 0}

In [ ]:
# Updating a single document:
index(
    updated_docs,
    record_manager,
    vectorstore,
    cleanup="incremental",
    source_id_key="source",
    key_encoder="blake2b",
)

{'num_added': 1, 'num_updated': 0, 'num_skipped': 0, 'num_deleted': 1}

In [ ]:
# Adding on a new document:
index(
    [Document(page_content="Digital marketing is a growing industry.", metadata={"source": "Wikipedia - Digital Marketing"})],
    record_manager,
    vectorstore,
    cleanup="incremental",
    source_id_key="source",
    key_encoder="blake2b",
)

{'num_added': 1, 'num_updated': 0, 'num_skipped': 0, 'num_deleted': 0}

Incremental API mode:
Deduplicates content for each document as you reingest it.

In [ ]:
# Skipping documents because the document hash is exactly the same:
index(
    [Document(page_content="Digital marketing is a growing industry.", metadata={"source": "Wikipedia - Digital Marketing"})],
    record_manager,
    vectorstore,
    cleanup="incremental",
    source_id_key="source",
    key_encoder="blake2b",
)

{'num_added': 0, 'num_updated': 0, 'num_skipped': 1, 'num_deleted': 0}

## There are 3 modes of using the Index API.


| Cleanup Mode | De-Duplicates Content | Parallelizable | Cleans Up Deleted Source Docs | Cleans Up Mutations of Source Docs and/or Derived Docs | Clean Up Timing      |
|--------------|-----------------------|----------------|-------------------------------|--------------------------------------------------------|----------------------|
| None         | ✅                     | ✅              | ❌                             | ❌                                                      | -                    |
| Incremental  | ✅                     | ✅              | ❌                             | ✅                                                      | Continuously         |
| Full         | ✅                     | ❌              | ✅                             | ✅                                                      | At end of indexing   |
